# 03c repair · 03c plus one AdaHome fix: a call that misses a required parameter becomes an ask

Built from `notebooks/03c_eval_fixes.ipynb`, which stays as it is, so 03c is the baseline. One change, in `infer.py`: when `env.validate` rejects an installed tool (read or write, gold asks for both) because a required or one-of parameter is missing, the agent makes `sys.ask_missing_info` for exactly those parameters and stops, instead of stopping with nothing. 214 of 217 gold asks name only such parameters. The ask is validated first, so a bad repair still costs nothing. AdaHome (arXiv 2607.18034) re-prompts the model on invalid output; this is the no-generation version, so it costs no time.

In the report, `repaired` is the new stop reason. Compare its count and score with `invalid` in a 03c run on the same inputs. Ceiling about 1 point: LFM3 had 12 invalid stops on public, 7 of them a missing parameter; Spark had 3 on held-out ASR, none of which an ask would have fixed.

## Vote (`VOTES`, added 2026-11-07)

`VOTES = 1` keeps everything above. With `VOTES = 5`, every step asks the same prompt with the tool list in 5 fixed orders at once (order 0 is the usual one; training shuffled the tools, so every order is in distribution) and keeps the most common call, compared with its free-text values blanked. Ties go to order 0, so 5 different answers give exactly the `VOTES = 1` answer. Why: on held-out, the 60.29 model's calls changed in 100 of 328 samples between gold and ASR text, so its decisions are unstable. This averages that out without a retrain.

The 5 prompts tokenise to the same length (480 of 480 public prompts checked), so they run as one batch with no padding. Time per step on a T4 is unmeasured, guess about 2x. If 5 prompts do not fit a T4, the step falls back to one prompt at a time. Each pred line gets `agree`: per step, [votes for the kept call, the order that gave it], and the report splits samples by whether the vote changed anything.

To measure: `bowen` + `thbased01`, `SMOKE = False`, `EVAL = True`, `VOTES = 5`, and keep only the `train gold` line of the run cell. Compare with gold 71.77 (unseen 69.14, seen 76.97). The Docker reads the same setting from its `VOTES` environment variable.

## Original 03c notes

Same pipeline as `03_eval_submit.ipynb`, which stays as it is. Three changes, none needs a retrain:
1. `TIME_BUDGET = 120`. The file upload is not timed, only the Docker is, and the old 25 s cap cut off long samples.
2. `STRIP_MARKS = True` removes the tagger's `[ ]` from ASR text. They leaked into arguments, and they are cleaner in training than at test.
3. Every line of `pred_*.jsonl` gets `stop` (why the agent stopped) and `raw` (model output per step). `submission.jsonl` keeps only its three keys.

**Do not copy `TIME_BUDGET = 120` into the Docker.** It must stay under the real 30 s limit there. This notebook tells you what the budget costs, it does not fix speed.

**Setup**
1. Settings: Accelerator **GPU T4 x2**.
2. Add Input: the private dataset, the output of exactly one of `01_asr_zipformer.ipynb` or `01b_asr_gipformer.ipynb`, and the output of exactly one of `02_llm_qwen.ipynb`, `02b_llm_spark.ipynb` or `02c_llm_lfm3.ipynb`. Use the same ASR output the LLM trained on.

**LFM3 was released on 2026-10-27, after the 2026-10-26 cutoff. Do not upload its `submission.jsonl` without written approval from the organisers.**

**Run**
1. Run all with `SMOKE = True`: 8 samples per GPU, to check the path.
2. Set `SMOKE = False`, then Save Version > Save & Run All. Expect about 3 hours with `EVAL = True`.
3. `EVAL = True` needs a checkpoint trained with `FINAL = False`. A `FINAL` model trained on the held-out samples, so its scores mean nothing: set `EVAL = False`.

**Reading the report**
- The previous eval scored gold 56.90 unseen and 59.48 seen, asr 40.08 unseen and 39.72 seen.
- `over 25 s` against `under 25 s` shows what the old budget was costing. Those samples still have to fit in 30 s in the Docker.
- `gold` against `asr` is the ASR cost, now without marks in the way. To price the marks alone, run once more with `STRIP_MARKS = False`.
- The `stop` lines split the rest. `unparsable` is a broken model output, `done` with fewer calls than gold is a model that gave up, and `invalid` is a schema-invalid call the agent refused to make.

**Output**: `submission.jsonl`, 5 uploads a day, and `pred_*.jsonl` for analysis.


In [ ]:
import glob
import json

import torch
assert torch.cuda.device_count() == 2, "Set Settings > Accelerator to GPU T4 x2"
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(2)])
_config = json.load(open(glob.glob("/kaggle/input/**/llm_model/config.json", recursive=True)[0], encoding="utf-8"))
TRANSFORMERS = {"spark2_5": "4.57.1", "lfm3": "5.20.2"}.get(_config["model_type"], "5.22.1")  # the version each model was trained on
print(_config["model_type"], "transformers", TRANSFORMERS)
!pip install -q "transformers=={TRANSFORMERS}" "accelerate==1.15.0"


In [ ]:
%%writefile settings.py

import glob
from pathlib import Path

SMOKE = True  # True: 8 samples per GPU
EVAL = True  # False for a FINAL model, nothing is held out
DTYPE = "float16"
SHARDS = 2  # one process per T4
MAX_STEPS = 8  # gold never uses more than 6 calls
MAX_NEW_TOKENS = 320  # the longest gold call is 304 characters, plus a written reason when the model was trained with WHY
TIME_BUDGET = 120.0  # a file upload is not timed. The Docker must stay under the real limit of 30 s
STRIP_MARKS = True  # remove the tagger's [ ] from ASR text. False: the old behaviour
SEED = 42
VOTES = 1  # >1: each step asks the tool list in VOTES fixed orders at once and keeps the most common call (ties: the usual order). 5 for the vote experiment

ROOT = Path(glob.glob("/kaggle/input/**/train/train.jsonl", recursive=True)[0]).parents[1]
LLM = str(Path(glob.glob("/kaggle/input/**/llm_model/config.json", recursive=True)[0]).parent)
LIFEMATE_DIR = str(Path(LLM).parent)  # lifemate.py as saved with these weights
ASR_JSON = glob.glob("/kaggle/input/**/asr_transcripts.json", recursive=True)[0]

In [ ]:
%%writefile infer.py
"""One shard of the agent: python infer.py <split> <gold|asr> <shard>. One process per split, each split has its own zaic_tools."""
import json
import random
import re
import sys
import time

import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # before torch: less fragmentation after a vote batch runs out of memory
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from transformers.integrations import sdpa_attention

from settings import *

sdpa_attention.use_gqa_in_sdpa = lambda *args, **kwargs: False  # a T4 has no grouped-key kernel: Qwen3.5 fell back to float32 math and ran out of memory on long prompts (public_0110, public_0157)

sys.path[:0] = [LIFEMATE_DIR, str(ROOT / sys.argv[1])]
import lifemate as lm
from zaic_tools import ToolEnv

split, mode, shard = sys.argv[1], sys.argv[2], int(sys.argv[3])
set_seed(SEED)
tok = AutoTokenizer.from_pretrained(LLM)
model = AutoModelForCausalLM.from_pretrained(LLM, dtype=DTYPE, device_map="auto" if torch.cuda.is_available() else None, trust_remote_code=True).eval()
tok.padding_side = "left"
tok.pad_token = tok.pad_token or lm.STOP  # padding=True needs one even when nothing is padded
stop_id = tok.convert_tokens_to_ids(lm.STOP)  # not every repo has a generation_config, so the default eos may never fire
catalog = lm.load_catalog(ROOT, split)
rows = lm.load_split(ROOT, split)
asr = json.load(open(ASR_JSON, encoding="utf-8")) if mode == "asr" else {}
if split == "train":
    rows = sum(lm.split_rows(rows)[1:], [])  # only held-out samples say anything
rows = rows[shard::SHARDS][: 8 if SMOKE else None]


TOOLS = "Danh sách tool:\n"
AGREE = []  # per step of the current sample: [votes for the kept call, the order that gave it]


def variant(msgs, k):
    """msgs with the tool list in its k-th fixed order; 0 is the order build_messages gave. Training shuffled the tools, so every order is in distribution."""
    if not k:
        return msgs
    head, sep, tools = msgs[0]["content"].partition(TOOLS)
    if not sep:
        raise SystemExit(f"VOTES > 1 needs {TOOLS!r} in the system prompt of this lifemate.py")  # not an Exception: run() must not swallow it sample by sample
    lines = tools.split("\n")
    random.Random(f"{k}|{msgs[0]['content']}").shuffle(lines)  # the same order at every step of a sample and in every run
    return [{**msgs[0], "content": head + TOOLS + "\n".join(lines)}] + msgs[1:]


def key(raw):
    """What the vote compares: the call with free-text (fuzzy) values blanked. Their wording differs between orders, the hash ignores it, and token F1 forgives it."""
    action = lm.parse_action(raw)
    if action is None:
        return "done" if '"done"' in raw else "unparsable"
    props = catalog.get(action[0], {}).get("parameters", {}).get("properties", {})
    return json.dumps([action[0], {k: "~" if props.get(k, {}).get("x-match") == "fuzzy" else v for k, v in action[1].items()}], sort_keys=True, ensure_ascii=False)


def pick(outs):
    """Plurality over the orders, ties to the lowest order: when every answer differs this is order 0, the VOTES = 1 answer."""
    keys = [key(o) for o in outs]
    best = max(keys, key=lambda k: (keys.count(k), -keys.index(k)))
    AGREE.append([keys.count(best), keys.index(best)])
    return outs[keys.index(best)]


def complete(texts):
    ids = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)  # tool orders tokenise to the same length (480 of 480 public prompts), so nothing is padded in practice
    assert ids.input_ids.shape[1] + MAX_NEW_TOKENS <= 16384, "over the competition context limit"
    out = model.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, eos_token_id=stop_id, pad_token_id=stop_id)
    return tok.batch_decode(out[:, ids.input_ids.shape[1]:], skip_special_tokens=True)


@torch.inference_mode()
def generate(msgs):
    texts = [lm.prompt_text(variant(msgs, k)) for k in range(VOTES)]
    try:
        return pick(complete(texts))
    except torch.cuda.OutOfMemoryError:  # ponytail: VOTES long prompts at once may not fit a T4 beside the weights; one at a time then, VOTES times slower
        pass  # retry outside the except block: inside it the error still holds the batch's GPU memory (train_0571 crashed that way)
    torch.cuda.empty_cache()
    return pick([complete([t])[0] for t in texts])


def clean(text):
    return re.sub(r"[\[\]]", "", text) if STRIP_MARKS else text  # norm() removes all punctuation, so every bracket here is a mark


def missing(name, args, installed):
    """The ask gold would make for an installed tool that misses a required or one-of parameter, else None. Reads too: gold asks for them 7 times (flights.search date). A one-of group names its first member, as gold does for ewallet.transfer (contact_id)."""
    tool = catalog.get(name)
    if tool is None or name.split(".")[0] not in installed:
        return None
    p = tool.get("parameters", {})
    empty = lambda k: args.get(k) in (None, "", [], {})  # zaic_tools counts these as missing
    need = [k for k in p.get("required", []) if empty(k)]
    if p.get("x-one-of") and all(any(empty(k) for k in grp) for grp in p["x-one-of"]):
        need.append(p["x-one-of"][0][0])
    return {"tool_name": name, "params": need} if need else None


def run(item):
    env = ToolEnv(split, item["id"], root=ROOT)  # exactly one per sample, every call on it goes into the submission
    texts = lm.gold_texts(item) if mode == "gold" else {c: clean(asr[f"{split}/{c}"]) for c in lm.clips(item)}
    msgs = lm.build_messages(item, texts, catalog)
    t0, invalid, failed, stop, raw = time.time(), 0, 0, "max_steps", []
    retried = False
    AGREE.clear()
    try:
        for _ in range(MAX_STEPS):
            if time.time() - t0 > TIME_BUDGET:
                stop = "budget"
                break
            raw.append(generate(msgs))
            action = lm.parse_action(raw[-1])
            if action is None:  # parse_action returns None for both, and it stays as saved with the weights
                stop = "done" if '"done"' in raw[-1] else "unparsable"
                break
            call = {"name": action[0], "arguments": action[1]}
            said = json.JSONDecoder().raw_decode(raw[-1][raw[-1].index("{"):])[0]  # the model's own JSON, so a trained "why" stays in context. Equals call for other models
            check = env.validate(*action)
            if not check["ok"]:  # validate is not logged: a call we never make costs no U penalty
                ask = missing(*action, item["context"]["installed_apps"])
                if ask and env.validate(lm.ASK, ask)["ok"]:  # gold ends on its ask too
                    env.call(lm.ASK, ask)
                    stop = "repaired"
                    break
                if not retried:  # one more try with the error as the result (4 of 5 invalid stops on public called an uninstalled contacts.search)
                    retried = True
                    msgs += [lm.asst(said), lm.result_msg(check)]
                    continue
                invalid, stop = 1, "invalid"
                break
            if env.get_tool_calls().count(call) >= 2:  # greedy loop, gold repeats a call at most once
                stop = "repeat"
                break
            if catalog.get(call["name"], {}).get("kind") == "read" and [c["name"] for c in env.get_tool_calls()].count(call["name"]) >= 2:  # gold reads one tool 3+ times in 1 of 1,545 samples; LFM3 ver3 retried empty searches 127 times on public, then sometimes wrote on a guess
                stop = "repeat"
                break
            result = env.call(*action)
            msgs += [lm.asst(said), lm.result_msg(result)]
            if "query" in action[1] and not catalog[call["name"]]["parameters"].get("required") and any(v == [] for v in result.values()):  # gold never reads an empty list (0 of 234 contacts.search): the query was misheard, so list everything once and let the model pick. 25 public samples hit this (contacts, event_tickets); a full list fits MAX_RESULT_CHARS
                msgs += [lm.asst({"name": call["name"], "arguments": {}}), lm.result_msg(env.call(call["name"], {}))]
            if call["name"] == lm.ASK:  # gold always ends here
                stop = "ask"
                break
    except Exception as e:  # keep the calls already made, a crash must not lose the whole line
        failed, stop = 1, "crash"
        print(item["id"], repr(e), flush=True)
    return {"id": item["id"], "tool_calls": env.get_tool_calls(), "state_hash": env.get_state_hash(), "sec": time.time() - t0, "invalid": invalid, "failed": failed, "stop": stop, "raw": raw, "agree": list(AGREE)}


with open(f"pred_{split}_{mode}_{shard}.jsonl", "w", encoding="utf-8") as f:
    for n, item in enumerate(rows):
        line = run(item)
        if split == "train":  # scoring only: the second env replays gold for its hash, nothing here reaches a submission
            gold = item["label"]["tool_calls"]
            ref = ToolEnv(split, item["id"], root=ROOT)
            for c in gold:
                ref.call(c["name"], c["arguments"])
            line["score"] = lm.score(line["tool_calls"], gold, line["state_hash"], ref.get_state_hash(), catalog)
            line["unseen"] = bool(lm.used_apps(item) & lm.HOLDOUT_APPS)
        f.write(json.dumps(line, ensure_ascii=False) + "\n")
        f.flush()
        if n % 25 == 0:
            print(split, mode, shard, n, "/", len(rows), f"{line['sec']:.1f}s", flush=True)

In [ ]:
from settings import *

print("LLM:", LLM, "\nASR:", ASR_JSON, "\nITN:", Path(ASR_JSON).with_name("vi_itn.py").exists())  # 01b leaves vi_itn.py beside its transcripts

if EVAL:
    !CUDA_VISIBLE_DEVICES=0 python infer.py train gold 0 & CUDA_VISIBLE_DEVICES=1 python infer.py train gold 1 & wait
    !CUDA_VISIBLE_DEVICES=0 python infer.py train asr 0 & CUDA_VISIBLE_DEVICES=1 python infer.py train asr 1 & wait
!CUDA_VISIBLE_DEVICES=0 python infer.py public_test asr 0 & CUDA_VISIBLE_DEVICES=1 python infer.py public_test asr 1 & wait

In [ ]:
import collections
import json
import statistics as st


def load(split, mode):
    return [json.loads(l) for f in sorted(glob.glob(f"pred_{split}_{mode}_*.jsonl")) for l in open(f, encoding="utf-8")]


def report(name, part):
    if part:
        s = sorted(x["sec"] for x in part)
        print(f"{name:20s} n={len(part):4d} score={100 * st.mean(x['score'] for x in part):6.2f} invalid={sum(x['invalid'] for x in part)} failed={sum(x['failed'] for x in part)} p95={s[int(.95 * len(s))]:.1f}s max={s[-1]:.1f}s")


if EVAL:
    for mode in ("gold", "asr"):
        p = load("train", mode)
        report(f"{mode} unseen", [x for x in p if x["unseen"]])
        report(f"{mode} seen", [x for x in p if not x["unseen"]])
        report(f"{mode} under 25 s", [x for x in p if x["sec"] <= 25])
        report(f"{mode} over 25 s", [x for x in p if x["sec"] > 25])
        for stop in sorted({x["stop"] for x in p}):
            report(f"{mode} {stop}", [x for x in p if x["stop"] == stop])  # repaired: was invalid in 03c
        votes = [a for x in p for a in x.get("agree", [])]
        if VOTES > 1 and votes:  # how often the vote kept another order than 0, and how often all orders agreed
            changed = [x for x in p if any(a[1] for a in x["agree"])]
            print(f"{mode} vote: {len(votes)} steps, {sum(a[1] > 0 for a in votes)} kept another order, {sum(a[0] == VOTES for a in votes)} unanimous")
            report(f"{mode} vote changed", changed)
            report(f"{mode} vote kept", [x for x in p if x not in changed])

In [ ]:
pred = load("public_test", "asr")
assert len({x["id"] for x in pred}) == len(pred)
expected = sum(1 for _ in open(ROOT / "public_test/public_test.jsonl", encoding="utf-8"))
assert SMOKE or len(pred) == expected, f"{len(pred)} lines, expected {expected}"
with open("submission.jsonl", "w", encoding="utf-8") as f:
    for x in pred:
        f.write(json.dumps({k: x[k] for k in ("id", "tool_calls", "state_hash")}, ensure_ascii=False) + "\n")
print(collections.Counter(x["stop"] for x in pred).most_common(), sum(x["sec"] > 30 for x in pred), "samples over 30 s")
for x in [x for x in pred if x["stop"] == "unparsable"][:5]:
    print(x["id"], repr(x["raw"][-1][:200]))
print(f"{len(pred)} lines, {sum(x['failed'] for x in pred)} failed, {sum(x['invalid'] for x in pred)} stopped on an invalid call, slowest {max(x['sec'] for x in pred):.1f}s")
!cd "{ROOT}/public_test" && PYTHONPATH=. python -m zaic_tools --split public_test --root .. --validate /kaggle/working/submission.jsonl